In [1]:

# STEP 1: Import Libraries

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity


# STEP 2: Load Twitter Dataset

file_path = r"F:\lp4\LP IV Updated Datasets\CBOW\twitter_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(df.head())
print("Columns:", df.columns)


# STEP 3: Select Tweet Column

possible_columns = ['text', 'tweet', 'tweets', 'content']

tweet_column = None

for col in df.columns:
    if col.lower() in possible_columns:
        tweet_column = col
        break

if tweet_column is None:
    text_columns = df.select_dtypes(include=['object']).columns

    if len(text_columns) == 0:
        raise ValueError("No text column found!")

    tweet_column = text_columns[0]

print("Using column:", tweet_column)


# STEP 4: Preprocess Tweets

def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()

    return words


df = df.dropna(subset=[tweet_column])

documents = df[tweet_column].astype(str).tolist()

tokenized_documents = [
    preprocess_text(doc) for doc in documents
]

# Remove empty documents

valid_data = [
    (doc, tokens)
    for doc, tokens in zip(documents, tokenized_documents)
    if len(tokens) > 0
]

documents = [item[0] for item in valid_data]
tokenized_documents = [item[1] for item in valid_data]

print("\nPreprocessed Tweets:")
print(tokenized_documents[:5])


# STEP 5: Train Word2Vec CBOW Model

model = Word2Vec(
    sentences=tokenized_documents,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,
    epochs=10
)

print("\nCBOW Model Training Completed!")


# STEP 6: Create Document Vector Function

def get_document_vector(words):

    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


# STEP 7: Create Vectors for All Documents

document_vectors = np.array([
    get_document_vector(doc)
    for doc in tokenized_documents
])

print("\nDocument Vectors Created!")
print("Document Vector Shape:", document_vectors.shape)


# STEP 8: Get User Input

query = input("\nEnter your search query: ")

query_tokens = preprocess_text(query)

query_vector = get_document_vector(query_tokens)

query_vector = query_vector.reshape(1, -1)


# STEP 9: Calculate Cosine Similarity

similarities = cosine_similarity(
    query_vector,
    document_vectors
)[0]

print("\nCosine Similarity Calculated!")


# STEP 10: Recommend Similar Documents

top_indices = np.argsort(similarities)[::-1][:3]

print("\nRecommended Tweets:")

for i in top_indices:

    print("\nTweet:", documents[i])
    print("Similarity Score:", similarities[i])


# STEP 11: Save CBOW Model

model.save("recommendation_cbow_model.model")

print("\nCBOW Model Saved Successfully!")


Dataset loaded successfully!
   Tweet_ID        Username  \
0         1         julie81   
1         2   richardhester   
2         3  williamsjoseph   
3         4     danielsmary   
4         5      carlwarren   

                                                Text  Retweets  Likes  \
0  Party least receive say or single. Prevent pre...         2     25   
1  Hotel still Congress may member staff. Media d...        35     29   
2  Nice be her debate industry that year. Film wh...        51     25   
3  Laugh explain situation career occur serious. ...        37     18   
4  Involve sense former often approach government...        27     80   

             Timestamp  
0  2023-01-30 11:00:51  
1  2023-01-02 22:45:58  
2  2023-01-18 11:25:19  
3  2023-04-10 22:06:29  
4  2023-01-24 07:12:21  
Columns: Index(['Tweet_ID', 'Username', 'Text', 'Retweets', 'Likes', 'Timestamp'], dtype='object')
Using column: Text

Preprocessed Tweets:
[['party', 'least', 'receive', 'say', 'or', 'single', '


Enter your search query:  artificial intelligence



Cosine Similarity Calculated!

Recommended Tweets:

Tweet: Body onto understand team about product beautiful. Might class picture guy significant resource material.
Minute begin happen adult.
Old carry pay government agree less night. Family maybe side process strategy choose. Until push respond agree close style as.
Similarity Score: 0.0

Tweet: Guess without successful save. Particular natural management on. Increase every large production decade ready once.
Must likely run piece act view young certainly. This look enjoy stage create soon natural statement.
Laugh material many reason artist never.
Similarity Score: 0.0

Tweet: You day agent likely region. Teacher data message build as third. Child forward her recent admit order black.
Trip source four into low. Table data him sing.
Plan assume land talk brother late less. Air effect reduce line.
Similarity Score: 0.0

CBOW Model Saved Successfully!
